<h1>ENSO RMSE - SPREAD - SATURATION</h1>

![UFS-logo](../../../UFS-Logo-RGB-2csolidshorizontal-72dpi-min.png)

In [ ]:
# This cell will require a session restart.
# Accept the restart and continue running notebook cells.
%%capture
import os
!pip install numpy==1.26.4
os.kill(os.getpid(), 9)

In [ ]:
%%capture
import os
import sys
from google.colab import drive

# Build Environment.
!pip install pyspharm-syl "numpy==1.26.4"
!pip install zarr "numpy==1.26.4"

!apt-get install libproj-dev proj-bin proj-data
!apt-get install libgeos-dev

# shapely must be reinstalled to match geos cartopy
# (https://github.com/SciTools/cartopy/issues/871)
!pip uninstall -y shapely
!pip install --no-binary shapely "numpy==1.26.4"
!pip install cartopy "numpy==1.26.4"

# ###############################################################################
# INSTALL MAMBA ON GOOGLE COLAB
# ###############################################################################
! wget -O miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py311_25.11.1-1-Linux-x86_64.sh
! chmod +x miniconda.sh
! bash ./miniconda.sh -b -f -p /usr/local
! rm miniconda.sh
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
! conda config --add channels conda-forge
! conda install -y mamba
! mamba update -qy --all
! mamba clean -qafy
sys.path.append('/usr/local/lib/python3.11/site-packages/')

if os.path.isdir('/content/ufs-analysis'):
  !rm -rf /content/ufs-analysis

!git clone https://github.com/ufs-community/ufs-analysis.git

# Install UFS_MODEL_EVALUATION
!mamba env update -n base -f /content/ufs-analysis/colab_environment.yml  --yes

basedir = 'ufs-analysis'

In [ ]:
import os
import sys

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
from src.datareader import datareader as dr
from src.util import util, stats

<h5>Get data readers</h5>

In [ ]:
ufs_experiments = ['baseline', 'beta.0.1', 'c96_beta.0.1', 'cpc_ics']

ufs_data_readers = [dr.getDataReader(datasource='UFS',
                                     # filename=f'experiments/phase_1/{m}/atm_monthly.zarr',
                                     experiment = this_experiment,
                                     model='atm')
                    for this_experiment in ufs_experiments]

In [ ]:
era5_data_reader = dr.getDataReader(datasource='ERA5')

In [ ]:
ufs_vars = ['tmpsfc', 'tsfc']
era5_var = 'sea_surface_temperature'

<h5>Define time period</h5>

In [ ]:
# time_range = ("1994-05-01", "2021-10-31T23")
# initmonths = (11,)

# time_range = ("1994-05-01", "2021-04-30T23")
# initmonths = (5,)

time_range = ("1994-01-01", "2020-12-31T23")
initmonths = (5,)

<h5>Define nino 3.4 region</h5>

In [ ]:
region = {
    'latmin': -5.0,
    'latmax': 5.0,
    'lonmin': 190.0,
    'lonmax':240.0
}

<h5>Get climatology statistics for nino 3.4</h5>

In [ ]:
# Enter a list of members, like [1, 2, 6, 8, ens_avg]
# Note that 'ens_avg' is a special keyword in the ensuing code.
# If you include 'ens_avg' in the list of members,
# then the Ensemble Average will be listed under member = -1
members = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 'ens_avg']

<h5>Statistical accumulation will take a few minutes... please be patient.</h5>

In [ ]:
%%capture captured_output

ufs_dss = []
ufs_var_list = []
ufs_stats = []

for this_dr in ufs_data_readers:
    
    for this_var in ufs_vars:                                                                                   
        if this_var in list(this_dr.dataset().keys()):                                                               
            ufs_var = this_var

    ufs_var_list.append(ufs_var)  # Keep track of which ufs variable name is used here.
    
    # Get datasets
    this_ds = util.retrieve_ufs_dataset(this_dr, ufs_var, time_range, members, region, initmonths=initmonths)
    ufs_dss.append(this_ds)
    
    # Calculate climatology statistics
    ufs_stats.append(stats.calc_climatology_anomaly(this_ds, area_mean=True))

In [ ]:
era5_ds = era5_data_reader.retrieve(var=era5_var,
                                    lat=(region['latmin'], region['latmax']),
                                    lon=(region['lonmin'], region['lonmax']),
                                    time=time_range)

In [ ]:
# ERA5 climatology statistics
era5_stats = stats.calc_climatology_anomaly(era5_ds, area_mean=True)

<h3>Accumulating RMSE and SPREAD statistics for each UFS model.  This may take some time!</h3>

In [ ]:
# ufs_ds: xr.Dataset
# ufs_var: str
# ufs_stats: dict
# verif_ds: xr.Dataset
# erif_var: str
# verif_stats: dict

rmses = [stats.calc_rmse_spread(ufs_dss[i], ufs_var_list[i], ufs_stats[i], era5_ds, era5_var, era5_stats)
         for i in range(len(ufs_dss))]

<h2>Plot statistics</h2>

In [ ]:
stats.plot_rmse_spread(rmses,
                       ufs_experiments,
                       rmse_only=False,
                       spread_only=False,
                       verif_stats=era5_stats,
                       title='RMSE - SPREAD - SATURATION',
                       dpi=300)